# Two-video room mesh: VGGT → COLMAP BA → CUDA MVS → mesh

This notebook starts with the 48-frame cross-video pilot. Do not run a larger tier until the pilot registers frames from both videos into one coherent reconstruction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/room_reconstruction')
INPUT_ZIP = DRIVE_ROOT / 'room_mesh_colab_input.zip'
WORK_ROOT = Path('/content/room_reconstruction')
VGGT_DIR = Path('/content/vggt')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
assert INPUT_ZIP.is_file(), f'Upload the prepared zip here first: {INPUT_ZIP}'
print('Input:', INPUT_ZIP, round(INPUT_ZIP.stat().st_size / 1e6, 1), 'MB')


In [ ]:
import subprocess, sys, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {gpu_name} ({gpu_gb:.1f} GB)')
if not VGGT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/vggt.git', str(VGGT_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(VGGT_DIR / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(VGGT_DIR / 'requirements_demo.txt')], check=True)
# Keep VGGT sparse export on its officially pinned PyCOLMAP 3.10 interface.
# PyCOLMAP 3.10 constructs Image(id=..., cam_from_world=...), while the resulting
# Python object exposes that identifier as image.image_id (there is no image.id property).
# Remove both distributions first because they install the same `pycolmap` module files.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'pycolmap-cuda12', 'pycolmap'], check=False)
# PyCOLMAP 3.10 was built for NumPy 1.x. Pin both together after all demo requirements.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
    'numpy==1.26.1', 'pycolmap==3.10.0',
], check=True)
def restore_vggt_pycolmap_310():
    converter = VGGT_DIR / 'vggt' / 'dependency' / 'np_to_pycolmap.py'
    backup = converter.with_suffix('.py.upstream')
    if backup.exists():
        converter.write_text(backup.read_text())
    source = converter.read_text()
    required = (
        'id=fidx + 1, name=',
        'cam_from_world=cam_from_world',
        'pycolmap.ListPoint2D(points2D_list)',
        'reconstruction.add_camera(camera)',
        'reconstruction.add_image(image)',
    )
    if not all(token in source for token in required):
        subprocess.run(
            ['git', '-C', str(VGGT_DIR), 'checkout', '--', 'vggt/dependency/np_to_pycolmap.py'],
            check=True,
        )
        source = converter.read_text()
    missing = [token for token in required if token not in source]
    if missing:
        raise RuntimeError(f'VGGT 3.10 converter restore failed; missing tokens: {missing}')
    print('VGGT converter restored for the official PyCOLMAP 3.10 API')

restore_vggt_pycolmap_310()
api_check = subprocess.run([
    sys.executable, '-c',
    "import importlib.metadata as metadata, pycolmap; print('Loaded:', pycolmap.__file__); print('Distribution:', metadata.version('pycolmap')); pose=pycolmap.Rigid3d(); camera=pycolmap.Camera(model='PINHOLE', width=16, height=16, params=[12.0, 12.0, 8.0, 8.0], camera_id=1); image=pycolmap.Image(id=7, name='probe', camera_id=1, cam_from_world=pose); print('Image identifier:', image.image_id); reconstruction=pycolmap.Reconstruction(); reconstruction.add_camera(camera); reconstruction.add_image(image); ok=(image.image_id == 7 and reconstruction.images[7].name == 'probe' and hasattr(pycolmap, 'ListPoint2D')); print('VGGT PyCOLMAP 3.10 legacy API:', 'OK' if ok else 'FAILED'); raise SystemExit(0 if ok else 2)",
], text=True, capture_output=True)
print(api_check.stdout, end='')
if api_check.returncode:
    print(api_check.stderr, end='')
    subprocess.run([sys.executable, '-m', 'pip', 'show', 'numpy', 'pycolmap', 'pycolmap-cuda12'], check=False)
    raise RuntimeError('PyCOLMAP isolation failed; copy the diagnostic output above.')


In [ ]:
import json, shutil, zipfile
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(INPUT_ZIP) as archive:
    archive.extractall(WORK_ROOT)
manifest = json.loads((WORK_ROOT / 'manifest.json').read_text())
print(json.dumps(manifest['scenes'], indent=2))
print('Usable filtered frames:', manifest['usable_frames'])


## Pilot: jointly solve both videos
The pilot is deliberately small enough for a T4. It is a validation gate, not the final mesh.

In [ ]:
import collections, os, shutil, subprocess, sys, time
def make_reduced_pilot(source_scene, target_scene, total_frames=24):
    images = sorted((source_scene / 'images').glob('*.jpg'))
    groups = collections.defaultdict(list)
    for image in images:
        groups[image.name.split('_', 1)[0]].append(image)
    target_images = target_scene / 'images'
    target_images.mkdir(parents=True, exist_ok=True)
    remaining = total_frames
    for group_index, (_, group) in enumerate(sorted(groups.items())):
        count = remaining if group_index == len(groups) - 1 else max(6, round(total_frames * len(group) / len(images)))
        count = min(count, len(group))
        indices = [round(i * (len(group) - 1) / max(1, count - 1)) for i in range(count)]
        for index in sorted(set(indices)):
            shutil.copy2(group[index], target_images / group[index].name)
        remaining -= count
    print('Reduced pilot:', len(list(target_images.glob('*.jpg'))), 'frames from', sorted(groups))
    return target_scene

def configure_vggt_fine_tracking_default():
    demo_path = VGGT_DIR / 'demo_colmap.py'
    source = demo_path.read_text()
    upstream = '"--fine_tracking", action="store_true", default=True'
    t4_safe = '"--fine_tracking", action="store_true", default=False'
    if upstream in source:
        backup = demo_path.with_suffix('.py.upstream')
        if not backup.exists():
            shutil.copy2(demo_path, backup)
        demo_path.write_text(source.replace(upstream, t4_safe, 1))
        print('Enabled explicit T4-safe coarse-tracking mode in demo_colmap.py')
    elif t4_safe not in source:
        raise RuntimeError('VGGT fine_tracking argument changed upstream; inspect demo_colmap.py before continuing.')

def verify_vggt_pycolmap_api():
    restore_vggt_pycolmap_310()
    check = subprocess.run(
        [sys.executable, '-c', "import importlib.metadata as metadata, pycolmap; pose=pycolmap.Rigid3d(); camera=pycolmap.Camera(model='PINHOLE', width=16, height=16, params=[12.0, 12.0, 8.0, 8.0], camera_id=1); image=pycolmap.Image(id=7, name='probe', camera_id=1, cam_from_world=pose); assert image.image_id == 7; reconstruction=pycolmap.Reconstruction(); reconstruction.add_camera(camera); reconstruction.add_image(image); assert reconstruction.images[7].name == 'probe'; assert hasattr(pycolmap, 'ListPoint2D'); print(metadata.version('pycolmap'))"],
        text=True, capture_output=True,
    )
    if check.returncode:
        raise RuntimeError(
            'Incompatible PyCOLMAP for VGGT sparse export. Rerun the dependency-install cell; '
            'it must report VGGT PyCOLMAP 3.10 legacy API: OK. Original error: ' + (check.stderr or check.stdout).strip()
        )
    print('Verified VGGT PyCOLMAP API:', check.stdout.strip())

def run_vggt_colmap(scene_dir, query_frames=5, query_points=2048, fine_tracking=None, use_ba=True):
    verify_vggt_pycolmap_api()
    configure_vggt_fine_tracking_default()
    if fine_tracking is None:
        fine_tracking = gpu_gb >= 22
    sparse = scene_dir / 'sparse'
    required = [sparse / name for name in ('cameras.bin', 'images.bin', 'points3D.bin')]
    mode_file = sparse / 'pipeline_mode.txt'
    if all(path.is_file() and path.stat().st_size > 0 for path in required):
        saved_mode = mode_file.read_text().strip() if mode_file.is_file() else 'unknown'
        if not use_ba or saved_mode == 'bundle_adjusted':
            print('Reusing', sparse, f'({saved_mode})')
            return
        print('Existing model is not bundle-adjusted; rebuilding for the requested mode.')
    if sparse.exists():
        failed = scene_dir / f'sparse_failed_{int(time.time())}'
        sparse.rename(failed)
        print('Preserved incomplete output at', failed)
    command = [
        sys.executable, 'demo_colmap.py', f'--scene_dir={scene_dir}',
        f'--max_query_pts={query_points}', f'--query_frame_num={query_frames}',
    ]
    if use_ba:
        command.append('--use_ba')
    if use_ba and fine_tracking:
        command.append('--fine_tracking')
    mode = ('BA + fine tracking' if fine_tracking else 'BA + coarse tracking') if use_ba else 'feed-forward (no BA, T4-safe)'
    print('Reconstruction mode:', mode)
    print(' '.join(map(str, command)))
    environment = os.environ.copy()
    environment['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    process = subprocess.Popen(
        command, cwd=VGGT_DIR, env=environment, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    tail = collections.deque(maxlen=120)
    for line in process.stdout:
        print(line, end='')
        tail.append(line.rstrip())
    return_code = process.wait()
    if return_code:
        failure_tail = '\n'.join(tail)
        if use_ba and fine_tracking and 'CUDA out of memory' in failure_tail:
            print('Fine tracking exceeded GPU memory; retrying once in coarse T4-safe mode.')
            torch.cuda.empty_cache()
            return run_vggt_colmap(
                scene_dir, query_frames=min(query_frames, 5),
                query_points=min(query_points, 1024), fine_tracking=False, use_ba=True,
            )
        ba_inlier_failure = 'No reconstruction can be built with BA' in failure_tail or 'Not enough inliers per frame' in failure_tail
        if use_ba and ba_inlier_failure:
            print('BA lacks cross-frame inliers; retrying with VGGT feed-forward poses and depth.')
            torch.cuda.empty_cache()
            return run_vggt_colmap(
                scene_dir, query_frames=query_frames, query_points=query_points,
                fine_tracking=False, use_ba=False,
            )
        print('GPU:', torch.cuda.get_device_name(0), f'{gpu_gb:.1f} GB')
        print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
        raise RuntimeError(
            f'VGGT/COLMAP failed with exit code {return_code}. Last output:\n' + '\n'.join(tail)
        )
    mode_file.write_text('bundle_adjusted' if use_ba else 'feedforward')
    print('Saved reconstruction mode:', mode_file.read_text())

PILOT_SCENE = WORK_ROOT / 'scene_pilot'
if gpu_gb < 20:
    PILOT_SCENE = make_reduced_pilot(PILOT_SCENE, WORK_ROOT / 'scene_pilot_24', total_frames=24)
pilot_query_points = 1024 if gpu_gb < 20 else 2048
pilot_query_frames = 5 if gpu_gb < 24 else 8
pilot_fine_tracking = gpu_gb >= 22
pilot_use_ba = gpu_gb >= 22
run_vggt_colmap(
    PILOT_SCENE, query_frames=pilot_query_frames,
    query_points=pilot_query_points, fine_tracking=pilot_fine_tracking, use_ba=pilot_use_ba,
)


In [ ]:
import collections, numpy as np, matplotlib.pyplot as plt, pycolmap
def validate_reconstruction(scene_dir, minimum_ratio=0.80):
    reconstruction = pycolmap.Reconstruction(str(scene_dir / 'sparse'))
    input_count = len(list((scene_dir / 'images').glob('*.jpg')))
    registered = list(reconstruction.images.values())
    ratio = len(registered) / max(1, input_count)
    by_video = collections.Counter(image.name.split('_', 1)[0] for image in registered)
    print('Registered:', len(registered), '/', input_count, f'({ratio:.1%})')
    print('By source video:', dict(by_video))
    print('Sparse points:', len(reconstruction.points3D))
    assert ratio >= minimum_ratio, 'Registration ratio is too low; do not continue.'
    assert by_video.get('01', 0) > 0 and by_video.get('02', 0) > 0, 'One video did not join the reconstruction.'
    centers = []
    for image in registered:
        pose = image.cam_from_world() if callable(image.cam_from_world) else image.cam_from_world
        centers.append(np.asarray(pose.inverse().translation))
    centers = np.asarray(centers)
    plt.figure(figsize=(7, 7))
    plt.scatter(centers[:, 0], centers[:, 2], s=12, alpha=.75)
    plt.axis('equal'); plt.grid(alpha=.2); plt.title('Pilot camera centers — inspect for one coherent room')
    plt.show()
    return reconstruction
pilot_reconstruction = validate_reconstruction(PILOT_SCENE)


## Larger solve (run only after the pilot looks correct)
T4 defaults to the pilot, L4/24 GB to the 96-frame medium set, and A100/40+ GB to the 220-frame full set. Override `SCENE_TIER` only if you understand the memory tradeoff.

In [ ]:
RUN_LARGER = False  # change to True only after the pilot passes
SCENE_TIER = 'scene_full' if gpu_gb >= 38 else ('scene_medium' if gpu_gb >= 22 else 'scene_pilot')
ACTIVE_SCENE = WORK_ROOT / SCENE_TIER
print('Selected tier:', SCENE_TIER)
if RUN_LARGER:
    run_vggt_colmap(
        ACTIVE_SCENE, query_frames=8, query_points=4096 if gpu_gb >= 22 else 1024,
        fine_tracking=gpu_gb >= 22, use_ba=gpu_gb >= 22,
    )
    active_reconstruction = validate_reconstruction(ACTIVE_SCENE, minimum_ratio=0.85)
else:
    ACTIVE_SCENE = PILOT_SCENE
    active_reconstruction = pilot_reconstruction


## CUDA dense reconstruction and Poisson mesh
Run this first on the validated pilot. PatchMatch requires a CUDA-enabled PyCOLMAP build.

In [ ]:
RUN_DENSE = False  # change to True after inspecting the trajectory above
if RUN_DENSE:
    dense = ACTIVE_SCENE / 'dense'
    fused = ACTIVE_SCENE / 'fused.ply'
    raw_mesh = ACTIVE_SCENE / 'room_poisson_raw.ply'
    web_mesh = ACTIVE_SCENE / 'room_poisson_web.ply'
    pycolmap.undistort_images(
        output_path=dense, input_path=ACTIVE_SCENE / 'sparse', image_path=ACTIVE_SCENE / 'images',
        undistort_options=pycolmap.UndistortCameraOptions(max_image_size=1600),
    )
    pycolmap.patch_match_stereo(
        dense, options=pycolmap.PatchMatchOptions(max_image_size=1600, gpu_index='0', cache_size=min(16.0, gpu_gb * .55)),
    )
    pycolmap.stereo_fusion(
        fused, dense, input_type='geometric', output_type='PLY',
        options=pycolmap.StereoFusionOptions(max_image_size=1600, cache_size=min(16.0, gpu_gb * .55)),
    )
    pycolmap.poisson_meshing(fused, raw_mesh, pycolmap.PoissonMeshingOptions(depth=11, trim=9.0, color=True))
    import trimesh
    mesh = trimesh.load(raw_mesh, process=False)
    ratio = min(1.0, 200000 / max(1, len(mesh.faces)))
    pycolmap.simplify_mesh(raw_mesh, web_mesh, pycolmap.MeshSimplificationOptions(target_face_ratio=ratio, interpolate_colors=True))
    print('Dense cloud:', fused, round(fused.stat().st_size / 1e6, 1), 'MB')
    print('Web mesh:', web_mesh, round(web_mesh.stat().st_size / 1e6, 1), 'MB')


In [ ]:
# Preserve the active reconstruction and previews in Drive.
RESULT_ZIP = DRIVE_ROOT / f'{ACTIVE_SCENE.name}_vggt_colmap_mesh.zip'
if RUN_DENSE:
    shutil.make_archive(str(RESULT_ZIP.with_suffix('')), 'zip', ACTIVE_SCENE)
    print('Saved:', RESULT_ZIP)
else:
    print('Set RUN_DENSE=True after the pilot trajectory is verified.')
